In [1]:
from datasets import load_dataset
from datetime import datetime
from collections import Counter

# Stream the dataset (does NOT download the full 89 GB)
# Streaming means we read row-by-row from Hugging Face on-the-fly
print("Setting up streaming connection to fddemarco/pushshift-reddit...")
ds = load_dataset("fddemarco/pushshift-reddit", split="train", streaming=True)

print("Streaming connection established.")
print(f"Dataset type: {type(ds)}")

d:\Shikha\bias_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting up streaming connection to fddemarco/pushshift-reddit...


d:\Shikha\bias_project\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhavi\.cache\huggingface\hub\datasets--fddemarco--pushshift-reddit. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Streaming connection established.
Dataset type: <class 'datasets.iterable_dataset.IterableDataset'>


In [2]:
# Stream the first 1000 records to verify what's in the dataset
# We're checking: which years? which subreddits? data quality?

from collections import Counter
from datetime import datetime

year_counts = Counter()
subreddit_counts = Counter()
sample_records = []

print("Streaming first 1000 records... this should take 30-60 seconds.\n")

for i, record in enumerate(ds):
    # Track the year via the created_utc timestamp
    year = datetime.fromtimestamp(record['created_utc']).year
    year_counts[year] += 1
    
    # Track which subreddit
    subreddit_counts[record['subreddit']] += 1
    
    # Save a few full records for inspection
    if i < 3:
        sample_records.append(record)
    
    if i >= 999:
        break
    
    # Progress update every 200 records
    if (i + 1) % 200 == 0:
        print(f"  Processed {i + 1} records...")

print(f"\n--- Sample of {sum(year_counts.values())} records ---\n")

print("Years found:")
for year in sorted(year_counts.keys()):
    print(f"  {year}: {year_counts[year]} records")

print(f"\nTop 15 subreddits in sample:")
for sub, count in subreddit_counts.most_common(15):
    print(f"  r/{sub}: {count}")

print(f"\nFirst record (full):")
for key, val in sample_records[0].items():
    val_str = str(val)[:100] if len(str(val)) > 100 else str(val)
    print(f"  {key}: {val_str}")

Streaming first 1000 records... this should take 30-60 seconds.

  Processed 200 records...
  Processed 400 records...
  Processed 600 records...
  Processed 800 records...

--- Sample of 1000 records ---

Years found:
  2012: 1000 records

Top 15 subreddits in sample:
  r/AdviceAnimals: 57
  r/funny: 52
  r/pics: 48
  r/AskReddit: 35
  r/fffffffuuuuuuuuuuuu: 32
  r/trees: 31
  r/gaming: 29
  r/Music: 25
  r/politics: 25
  r/WTF: 24
  r/SteamGameSwap: 21
  r/videos: 20
  r/tf2trade: 15
  r/circlejerk: 14
  r/Fitness: 10

First record (full):
  author: doopercooper
  created_utc: 1325462399
  id: nz3g9
  num_comments: 24
  score: 82
  selftext: 
  subreddit: videos
  subreddit_id: t5_2qh1e
  title: guy chases after runaway car and barely saves it before it enters a busy street


In [3]:
# Skip ahead and sample records at different points in the dataset
# to find out the actual time range

from collections import Counter
from datetime import datetime

print("Skipping forward to sample records at different points...\n")

# Re-create the stream (skip + take doesn't work on iterables reliably)
ds_check = load_dataset("fddemarco/pushshift-reddit", split="train", streaming=True)

# Sample at three checkpoints: 1M, 100M, and 500M rows in
checkpoints = [1_000_000, 100_000_000, 500_000_000]
samples_at = {}

current_checkpoint_idx = 0
target = checkpoints[0]

for i, record in enumerate(ds_check):
    if i == target:
        year = datetime.fromtimestamp(record['created_utc']).year
        month = datetime.fromtimestamp(record['created_utc']).month
        samples_at[target] = (year, month, record['subreddit'])
        print(f"  Record {target:,}: {year}-{month:02d}, r/{record['subreddit']}")
        
        current_checkpoint_idx += 1
        if current_checkpoint_idx >= len(checkpoints):
            break
        target = checkpoints[current_checkpoint_idx]

print(f"\nDataset time range based on checkpoints:")
print(samples_at)

Skipping forward to sample records at different points...

  Record 1,000,000: 2012-01, r/beer


KeyboardInterrupt: 

In [6]:
import requests
from datetime import datetime

# PullPush API endpoint for submissions
# This pulls posts from r/california for March 2020
url = "https://api.pullpush.io/reddit/search/submission/"

# UNIX timestamps for March 2020
# March 1, 2020 00:00 UTC = 1583020800
# April 1, 2020 00:00 UTC = 1585699200
params = {
    "subreddit": "california",
    "after": 1583020800,
    "before": 1585699200,
    "size": 10  # just get 10 for the test
}

print(f"Querying PullPush for r/california, March 2020...")
print(f"URL: {url}")
print(f"Params: {params}\n")

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}")
print(f"Content type: {response.headers.get('Content-Type')}\n")

if response.status_code == 200:
    data = response.json()
    posts = data.get('data', [])
    print(f"Posts returned: {len(posts)}\n")
    
    if posts:
        print(f"First post fields available:")
        for key in posts[0].keys():
            print(f"  - {key}")
        
        print(f"\nFirst 3 posts:\n")
        for i, post in enumerate(posts[:3], 1):
            print(f"Post {i}:")
            print(f"  Title:     {post.get('title', '(no title)')[:80]}")
            print(f"  Author:    {post.get('author', '(unknown)')}")
            print(f"  Subreddit: {post.get('subreddit', '(unknown)')}")
            print(f"  Created:   {datetime.fromtimestamp(post.get('created_utc', 0))}")
            print(f"  Score:     {post.get('score', 0)}")
            print(f"  Selftext:  {(post.get('selftext') or '')[:200]}")
            print()
    else:
        print("No posts returned. This could mean:")
        print("- No posts in that subreddit for that time range")
        print("- Subreddit doesn't exist in PullPush's data")
        print("- API issue")
else:
    print(f"Error response: {response.text[:500]}")

Querying PullPush for r/california, March 2020...
URL: https://api.pullpush.io/reddit/search/submission/
Params: {'subreddit': 'california', 'after': 1583020800, 'before': 1585699200, 'size': 10}

Status: 200
Content type: application/json

Posts returned: 10

First post fields available:
  - all_awardings
  - allow_live_comments
  - archived
  - author
  - author_created_utc
  - author_flair_background_color
  - author_flair_css_class
  - author_flair_richtext
  - author_flair_template_id
  - author_flair_text
  - author_flair_text_color
  - author_flair_type
  - author_fullname
  - author_patreon_flair
  - author_premium
  - awarders
  - can_gild
  - can_mod_post
  - category
  - content_categories
  - contest_mode
  - created_utc
  - discussion_type
  - distinguished
  - domain
  - edited
  - gilded
  - gildings
  - hidden
  - id
  - is_crosspostable
  - is_meta
  - is_original_content
  - is_reddit_media_domain
  - is_robot_indexable
  - is_self
  - is_video
  - link_flair_backgrou

In [7]:
# Re-use the data variable from the previous cell
posts = data.get('data', [])

empty_selftext = 0
deleted_selftext = 0
real_selftext = 0
link_only = 0  # posts with URL but no selftext (link posts)

for post in posts:
    selftext = (post.get('selftext') or '').strip()
    
    if selftext == '[deleted]' or selftext == '[removed]':
        deleted_selftext += 1
    elif selftext == '':
        link_only += 1
    elif len(selftext) > 0:
        real_selftext += 1

print(f"Of {len(posts)} posts:")
print(f"  With real selftext content: {real_selftext}")
print(f"  Link posts (empty selftext): {link_only}")
print(f"  Deleted/removed selftext:    {deleted_selftext}")
print(f"\nAll posts have titles regardless - title is usable text for embeddings.")

Of 10 posts:
  With real selftext content: 0
  Link posts (empty selftext): 2
  Deleted/removed selftext:    8

All posts have titles regardless - title is usable text for embeddings.


In [8]:
# Try a larger sample, sorted by score (highest first) to see if better-quality posts retain text
url = "https://api.pullpush.io/reddit/search/submission/"

params = {
    "subreddit": "california",
    "after": 1583020800,
    "before": 1585699200,
    "size": 100,  # larger sample
    "sort": "score",  # sort by score descending
    "sort_type": "score"
}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    posts = data.get('data', [])
    
    empty_selftext = 0
    deleted_selftext = 0
    real_selftext = 0
    link_only = 0
    total_text_chars = 0
    
    for post in posts:
        selftext = (post.get('selftext') or '').strip()
        title = (post.get('title') or '').strip()
        
        if selftext in ('[deleted]', '[removed]'):
            deleted_selftext += 1
        elif selftext == '':
            link_only += 1
        else:
            real_selftext += 1
            total_text_chars += len(selftext)
        
        total_text_chars += len(title)
    
    print(f"\nOf {len(posts)} posts (sorted by score):")
    print(f"  With real selftext content: {real_selftext}")
    print(f"  Link posts (empty selftext): {link_only}")
    print(f"  Deleted/removed selftext:    {deleted_selftext}")
    print(f"\nTotal usable text (titles + non-deleted selftext): {total_text_chars} chars")
    print(f"Average chars per post: {total_text_chars / len(posts):.0f}")

Status: 200

Of 100 posts (sorted by score):
  With real selftext content: 4
  Link posts (empty selftext): 96
  Deleted/removed selftext:    0

Total usable text (titles + non-deleted selftext): 33477 chars
Average chars per post: 335


In [9]:
# Try the same time/subreddit but for COMMENTS instead of submissions
url = "https://api.pullpush.io/reddit/search/comment/"

params = {
    "subreddit": "california",
    "after": 1583020800,
    "before": 1585699200,
    "size": 20
}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}\n")

if response.status_code == 200:
    data = response.json()
    comments = data.get('data', [])
    
    deleted_count = 0
    real_count = 0
    total_chars = 0
    
    for c in comments:
        body = (c.get('body') or '').strip()
        if body in ('[deleted]', '[removed]'):
            deleted_count += 1
        else:
            real_count += 1
            total_chars += len(body)
    
    print(f"Of {len(comments)} comments:")
    print(f"  With real body: {real_count}")
    print(f"  Deleted/removed: {deleted_count}")
    print(f"  Total real comment text: {total_chars} chars")
    print(f"\nFirst real comment (if any):")
    for c in comments:
        body = (c.get('body') or '').strip()
        if body not in ('[deleted]', '[removed]', ''):
            print(f"  {body[:300]}")
            break

Status: 200

Of 20 comments:
  With real body: 14
  Deleted/removed: 6
  Total real comment text: 5781 chars

First real comment (if any):
  Yes that's in the article

&gt; Unlike the U.S., where tests have been in short supply, Asian cities had the advantages of being able to test for the virus widely, isolating the infected, finding their close contacts, and quarantining those who may have been infected.

&gt; [Wide availability of tes


In [4]:
from huggingface_hub import HfApi

api = HfApi()

# List all files in this dataset
files = api.list_repo_files("fddemarco/pushshift-reddit", repo_type="dataset")

# Filter to parquet files (the data files)
parquet_files = [f for f in files if f.endswith('.parquet')]

print(f"Total files in dataset: {len(files)}")
print(f"Parquet files: {len(parquet_files)}\n")

print("First 10 parquet files:")
for f in parquet_files[:10]:
    print(f"  {f}")

print("\nLast 10 parquet files:")
for f in parquet_files[-10:]:
    print(f"  {f}")

Total files in dataset: 220
Parquet files: 218

First 10 parquet files:
  data/RS_2012-01_00.parquet
  data/RS_2012-01_01.parquet
  data/RS_2012-01_02.parquet
  data/RS_2012-02_00.parquet
  data/RS_2012-02_01.parquet
  data/RS_2012-02_02.parquet
  data/RS_2012-03_00.parquet
  data/RS_2012-03_01.parquet
  data/RS_2012-03_02.parquet
  data/RS_2012-04_00.parquet

Last 10 parquet files:
  data/RS_2018-03_00.parquet
  data/RS_2018-04_00.parquet
  data/RS_2018-05_00.parquet
  data/RS_2018-06_00.parquet
  data/RS_2018-07_00.parquet
  data/RS_2018-08_00.parquet
  data/RS_2018-09_00.parquet
  data/RS_2018-10_00.parquet
  data/RS_2018-11_00.parquet
  data/RS_2018-12_00.parquet


In [4]:
import sys
print("Python executable being used by notebook:")
print(sys.executable)

Python executable being used by notebook:
d:\Shikha\bias_project\venv\Scripts\python.exe


In [10]:
import requests
from time import sleep

# Mix of states: big/medium/small, different regions
TEST_STATES = [
    ('california', 'big', 'west'),
    ('texas', 'big', 'south'),
    ('newyork', 'big', 'northeast'),
    ('ohio', 'medium', 'midwest'),
    ('arizona', 'medium', 'west'),
    ('alabama', 'medium', 'south'),
    ('wyoming', 'small', 'west'),
    ('vermont', 'small', 'northeast'),
    ('northdakota', 'small', 'midwest'),
]

# Test window: March 2020 (active period during early COVID)
AFTER = 1583020800   # March 1, 2020
BEFORE = 1585699200  # April 1, 2020

results = []

for subreddit, size_cat, region in TEST_STATES:
    print(f"\n--- r/{subreddit} ({size_cat}, {region}) ---")
    
    # 1. Check submissions count
    sub_url = "https://api.pullpush.io/reddit/search/submission/"
    sub_params = {
        "subreddit": subreddit,
        "after": AFTER,
        "before": BEFORE,
        "size": 100,
        "sort": "score",
        "sort_type": "score"
    }
    
    try:
        r = requests.get(sub_url, params=sub_params, timeout=30)
        sub_data = r.json().get('data', []) if r.status_code == 200 else []
        sub_count = len(sub_data)
        sub_text = sum(len((p.get('title') or '') + (p.get('selftext') or '')) 
                       for p in sub_data 
                       if (p.get('selftext') or '') not in ('[deleted]', '[removed]'))
    except Exception as e:
        sub_count = 0
        sub_text = 0
        print(f"  Submissions error: {e}")
    
    sleep(1)  # be polite to the API
    
    # 2. Check comments count
    com_url = "https://api.pullpush.io/reddit/search/comment/"
    com_params = {
        "subreddit": subreddit,
        "after": AFTER,
        "before": BEFORE,
        "size": 100
    }
    
    try:
        r = requests.get(com_url, params=com_params, timeout=30)
        com_data = r.json().get('data', []) if r.status_code == 200 else []
        com_count = len(com_data)
        com_real = [c for c in com_data 
                    if (c.get('body') or '') not in ('[deleted]', '[removed]', '')]
        com_text = sum(len(c.get('body') or '') for c in com_real)
    except Exception as e:
        com_count = 0
        com_text = 0
        com_real = []
        print(f"  Comments error: {e}")
    
    sleep(1)  # be polite to the API
    
    print(f"  Submissions: {sub_count} returned, {sub_text} usable chars")
    print(f"  Comments:    {com_count} returned, {len(com_real)} non-deleted, {com_text} usable chars")
    
    results.append({
        'subreddit': subreddit,
        'size': size_cat,
        'region': region,
        'submissions': sub_count,
        'sub_text': sub_text,
        'comments': com_count,
        'comments_real': len(com_real),
        'com_text': com_text,
    })

print("\n\n--- SUMMARY ---")
print(f"{'Subreddit':<16} {'Size':<8} {'Region':<10} {'Subs':<6} {'Comms':<6} {'Real':<6} {'Text(K)':<8}")
print("-" * 70)
for r in results:
    text_k = (r['sub_text'] + r['com_text']) / 1000
    print(f"r/{r['subreddit']:<14} {r['size']:<8} {r['region']:<10} "
          f"{r['submissions']:<6} {r['comments']:<6} {r['comments_real']:<6} "
          f"{text_k:.1f}")


--- r/california (big, west) ---
  Submissions: 100 returned, 33477 usable chars
  Comments:    100 returned, 80 non-deleted, 19324 usable chars

--- r/texas (big, south) ---
  Submissions: 100 returned, 45141 usable chars
  Comments:    100 returned, 94 non-deleted, 15717 usable chars

--- r/newyork (big, northeast) ---
  Submissions: 96 returned, 28166 usable chars
  Comments:    100 returned, 94 non-deleted, 13521 usable chars

--- r/ohio (medium, midwest) ---
  Submissions: 100 returned, 69281 usable chars
  Comments:    100 returned, 87 non-deleted, 12707 usable chars

--- r/arizona (medium, west) ---
  Submissions: 100 returned, 8693 usable chars
  Comments:    100 returned, 80 non-deleted, 17942 usable chars

--- r/alabama (medium, south) ---
  Submissions: 100 returned, 52514 usable chars
  Comments:    100 returned, 75 non-deleted, 11260 usable chars

--- r/wyoming (small, west) ---
  Submissions: 100 returned, 8302 usable chars
  Comments:    100 returned, 92 non-deleted, 13